<a href="https://colab.research.google.com/github/uyx7uy/ds2002-fa26/blob/main/2026-10-02%20-%20Normalize%20and%20Join%20Nested%20JSON%20-%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** One row of my result is one line item from an order.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']

print(unmatched[['vendor_id', 'qty']])
print('Unmatched units:', unmatched['qty'].sum())

joined

  vendor_id  qty
3      V-07    3
Unmatched units: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined['vendor_name'] = joined['name'].fillna('Unknown vendor')

vendor_totals = joined.groupby('vendor_name')['qty'].sum()
vendor_totals = vendor_totals.sort_values(ascending=False)

# I kept the unmatched vendor as Unknown vendor because it still sold items.
vendor_totals

,qty
vendor_name,
Hoos Burgers,3
Unknown vendor,3
Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = joined.groupby('zone')['qty'].sum()
zone_totals = zone_totals.sort_values(ascending=False)

print(zone_totals)
print('Zone total:', zone_totals.sum())
print('Missing from zone total:', joined['qty'].sum() - zone_totals.sum())

zone
A    4
Name: qty, dtype: int64
Zone total: 4
Missing from zone total: 3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},
]

ragged_with_lines = []

for order in ragged_json:
    if 'lines' in order:
        ragged_with_lines.append(order)

ragged_lines = pd.json_normalize(ragged_with_lines, record_path='lines', meta=['order', 'vendor_id'])

ragged_lines['qty_missing'] = ragged_lines['qty'].isna()

ragged_lines

,item,qty,order,vendor_id,qty_missing
0,Soda,2.0,4,V-01,False
1,Fries,NaN,6,V-01,True


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is useful because each item and quantity can be counted separately. This makes it easy to total quantities by vendor or zone. If there was only one row per order, the items would still be nested inside a list, so grouping and summing would be harder.

b) A missing quantity means the data was not provided, while a quantity of zero means we know that zero items were sold. If they are treated as the same thing, totals and averages can become misleading because missing data would be counted like real zero sales.